# Ачча! — озвучка фраз (Indic Parler-TTS)

Блокнот озвучивает фразы приложения «Ачча!» открытой моделью **Indic Parler-TTS** (AI4Bharat, лицензия Apache 2.0) и сохраняет mp3 в ваш Google Drive.

**Перед запуском:** меню **Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель: T4 GPU → Сохранить**.

Дальше запускайте ячейки по порядку (кнопка ▶ слева от ячейки). Шаг 5 — пробное прослушивание: если голос не понравился, поменяйте имя голоса на шаге 2 и запустите шаги 2 и 5 ещё раз. Шаг 6 озвучивает всё (10–20 минут).

## 1. Установка (2–3 минуты)

В конце сеанс перезапустится сам и появится сообщение о сбое — это нормально. Дальше начинайте с шага 2, шаг 1 повторять не нужно.

In [ ]:
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile
# parler-tts ставит старую версию protobuf, из-за неё не загружается transformers — возвращаем новую
!pip -q install -U "protobuf>=5.28"
!apt -qq install -y ffmpeg > /dev/null
print('Готово. Сеанс сейчас перезапустится сам — это нормально. После перезапуска начинайте с шага 2.')
import os; os.kill(os.getpid(), 9)

## 2. Настройки

Для английского создатели модели рекомендуют голоса **Thoma** (мужской) и **Mary** (женский); полный список из 21 голоса — на странице модели huggingface.co/ai4bharat/indic-parler-tts.
- `LOCAL_VOICE` — голос для реплик индийцев (индийский английский)
- `TOURIST_VOICE` — голос для реплик туриста
- `ONLY_LOCAL = True` — озвучить только реплики индийцев (туристу останется голос телефона)

In [ ]:
PHRASES_URL = 'https://achha-india.github.io/app/phrases.json'

LOCAL_VOICE = 'Thoma'
TOURIST_VOICE = 'Mary'
ONLY_LOCAL = False

LOCAL_STYLE = "{name} speaks in a clear Indian English accent at a moderate pace, with a friendly tone. The recording is very clear and close, with no background noise."
TOURIST_STYLE = "{name} speaks slowly and clearly with a neutral accent, in a calm tone. The recording is very clear and close, with no background noise."

OUT_DIR = '/content/drive/MyDrive/Achha/audio'
print('Настройки сохранены')

## 3. Загрузка модели (3–5 минут)

Если появится ошибка про доступ (401 / gated), откройте страницу модели huggingface.co/ai4bharat/indic-parler-tts, войдите в бесплатный аккаунт Hugging Face, нажмите «Agree and access» и добавьте токен в Colab: значок ключа слева → `HF_TOKEN`.

In [ ]:
import torch
from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get('HF_TOKEN'))
    print('Вход в Hugging Face выполнен')
except Exception as e:
    print('Не удалось прочитать секрет HF_TOKEN:', e)
    print('Проверьте: имя секрета ровно HF_TOKEN и включён «Доступ из блокнотов»')
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device, '(если cpu — включите T4 GPU, см. начало)')
MODEL_ID = 'ai4bharat/indic-parler-tts'
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR = model.config.sampling_rate
print('Модель загружена')

## 4. Фразы с сайта

In [ ]:
import json, urllib.request
data = json.load(urllib.request.urlopen(PHRASES_URL))
lines = [l for st in data['stages'] for s in st['situations'] for l in s['lines']]
todo = [l for l in lines if l['role'] == 'local' or not ONLY_LOCAL]
print('Всего реплик:', len(lines), '| к озвучке:', len(todo))

import soundfile as sf, numpy as np, subprocess, os

def synth(text, role, seed=7):
    name = LOCAL_VOICE if role == 'local' else TOURIST_VOICE
    style = (LOCAL_STYLE if role == 'local' else TOURIST_STYLE).format(name=name)
    torch.manual_seed(seed)
    d = description_tokenizer(style, return_tensors='pt').to(device)
    p = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                           prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    a = g.cpu().numpy().squeeze().astype(np.float32)
    # обрезаем тишину по краям
    nz = np.where(np.abs(a) > 0.01)[0]
    if len(nz): a = a[max(0, nz[0] - int(0.05 * SR)): nz[-1] + int(0.15 * SR)]
    return a

def to_mp3(audio, path):
    wav = path.replace('.mp3', '.wav')
    sf.write(wav, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-ac', '1', '-b:a', '64k', path], check=True)
    os.remove(wav)
print('Готово')

## 5. Пробное прослушивание

In [ ]:
from IPython.display import Audio, display
samples = [l for l in todo if l['id'] in ('market-02', 'road-13', 'market-01')] or todo[:3]
for l in samples:
    print(l['id'], '·', 'индиец' if l['role'] == 'local' else 'турист', '·', l['en'])
    display(Audio(synth(l['en'], l['role']), rate=SR))

## 6. Озвучить всё и сохранить в Google Drive

Colab попросит доступ к Google Drive — разрешите. Файлы появятся в папке **Achha/audio**, а рядом архив **achha-audio.zip**. Этот архив пришлите в чат.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
done, failed = 0, []
for i, l in enumerate(todo, 1):
    path = f"{OUT_DIR}/{l['id']}.mp3"
    if os.path.exists(path):
        done += 1; continue
    try:
        to_mp3(synth(l['en'], l['role']), path)
        done += 1
        print(f"[{i}/{len(todo)}] {l['id']} ✓")
    except Exception as e:
        failed.append(l['id']); print(f"[{i}/{len(todo)}] {l['id']} ошибка: {e}")
import shutil
zip_path = shutil.make_archive('/content/drive/MyDrive/Achha/achha-audio', 'zip', OUT_DIR)
print('\nГотово:', done, '| ошибки:', failed or 'нет')
print('Архив:', zip_path)
from google.colab import files
files.download(zip_path)

**Переозвучить отдельные фразы:** удалите их mp3 из папки Achha/audio на Drive и запустите шаг 6 ещё раз — остальные файлы он пропустит. Можно также поменять `seed` в функции `synth` — интонация станет немного другой.